## ② 프롬프트 3종 × 모델 2종 비교

### 준비
- [ ] 상용 API 모델 하나 선정
- [ ] 오픈 LLM 선정: 포르투갈어가 되는 소형 instruct 모델
- [ ] 오픈 LLM을 vLLM/Ollama 등으로 Docker에서 서빙. OpenAI 호환 엔드포인트를 쓰면 두 모델을 같은 코드로 호출할 수 있다.
- [ ] 다국어 임베딩 모델로 예시 풀을 벡터화해 pgvector에 적재

### 실험 설계

세 프롬프트 모두 예시 수 k, 출력 형식(JSON), temperature=0을 동일하게 맞춘다.

| 프롬프트 | 내용 |
|---|---|
| zero-shot | 카테고리 정의만 제공 |
| 고정 few-shot | 풀에서 고른 대표 예시 k개를 고정 |
| RAG few-shot | 리뷰마다 유사도 top-k 예시를 검색 |
| (권장) 랜덤 few-shot | 리뷰마다 풀에서 랜덤 k개 — 대조군 |

**효과 분리**
- 예시 효과 = 고정 few-shot − zero-shot
- 검색 효과 = RAG − 고정 few-shot
- RAG와 고정 few-shot은 "리뷰마다 예시가 바뀐다"는 점과 "비슷한 예시다"라는 점이 동시에 다르다. 랜덤 few-shot 대조군과 비교해야 순수한 유사도 효과를 분리할 수 있다.

### 지표
- **정확도:** macro-F1, 클래스별 F1, 혼동행렬, JSON 파싱 실패율
- **비용:** API는 토큰 × 단가, 로컬은 GPU 시간 환산
- **속도:** 지연시간 p50/p95, 처리량
- 평가셋이 작으므로 부트스트랩 신뢰구간을 함께 보고한다.

### 마무리
- [ ] 설정 파일로 돌아가는 실험 스크립트 작성, 결과표 CSV 저장
- [ ] 최종 조합을 골라 대상 전체(약 1만 건) 분류

**산출물:** 비교 결과표, 선택 근거, 전체 예측 결과

In [ ]:
# !pip install ollama

  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 452.9 kB/s  0:00:06 eta 0:00:01
Using cached annotated_types-0.8.0-py3-none-any.whl (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [ollama]10/11 [ollama]c]


In [10]:
# ===== 라이브러리 =====
import json
import time
import re

import ollama
import polars as pl

from pathlib import Path

# ===== 경로 설정 =====
DATA_DIR = "../data"
processed_dir = Path(DATA_DIR) / "processed"

# 분류 테스트

In [13]:
# ── 설정: 모델별 옵션과 라벨 (OUTPUT_SCHEMA는 삭제) ──
MODEL_CONFIGS = {
    "qwen3.5:4b": {"think": False},
    "gemma4:e4b-it-qat": {},
}

LABELS = ["배송 지연", "누락", "제품 오배송", "제품 불량", "위조·비정품", "제품 불만족", "기타"]

# 연결 확인용 임시 프롬프트 (정식 프롬프트는 다음 단계에서 설계)
TEST_PROMPT = (
    "브라질 이커머스의 포르투갈어 불만 리뷰를 읽고, 불만의 주 원인을 "
    f"다음 라벨 중 하나로 분류하라: {', '.join(LABELS)}\n"
    '반드시 {"label": "라벨"} 형식의 JSON 한 줄로만 답하라.'
)

In [4]:
# ── 테스트용 리뷰 1건 불러오기 ──
# 경로는 실제 위치에 맞게 수정
pool_df = pl.read_csv(processed_dir / "pool.csv")
# utf-8-sig로 저장돼 첫 컬럼명에 BOM이 붙을 수 있어 제거
pool_df = pool_df.rename(lambda col_name: col_name.lstrip("\ufeff"))

sample_row = pool_df.row(0, named=True)
print(sample_row["label"], "|", sample_row["review_text"])

배송 지연 | Ainda não recebi o meu pedido, estava programado para o dia 13/12 e até agora nada, hoje já é dia 17/12


In [11]:
# ── 응답 파싱: JSON 부분만 추출 → 라벨 검증 ──
# 모델이 ```json 코드블록이나 설명 문장을 붙이는 경우를 대비해 첫 { ... } 만 꺼냄
def parse_label(raw_text):
    json_match = re.search(r"\{.*?\}", raw_text, re.DOTALL)
    if json_match is None:
        return None
    try:
        label = json.loads(json_match.group()).get("label")
    except json.JSONDecodeError:
        return None
    # 7개 라벨 중 하나가 아니면 실패로 처리
    return label if label in LABELS else None


# ── 분류 요청: 결과를 dict 하나로 반환 ──
def classify_review(model_name, system_prompt, review_text):
    start_time = time.perf_counter()
    response = ollama.chat(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": review_text},
        ],
        options={"temperature": 0},
        **MODEL_CONFIGS[model_name],
    )
    elapsed_sec = time.perf_counter() - start_time

    raw_output = response.message.content
    predicted_label = parse_label(raw_output)
    return {
        "model": model_name,
        "pred_label": predicted_label,
        "parse_ok": predicted_label is not None,  # 파싱 실패율 계산용
        "raw_output": raw_output,                 # 실패 원인 확인용 원문
        "elapsed_sec": round(elapsed_sec, 2),
        "input_tokens": response.prompt_eval_count,
        "output_tokens": response.eval_count,
    }

In [15]:
# ── 테스트: 모델 하나씩, 워밍업 1회 후 측정 ──
test_results = []
for model_name in MODEL_CONFIGS:
    classify_review(model_name, TEST_PROMPT, sample_row["review_text"])  # 워밍업 (로딩 시간 제외)
    test_results.append(classify_review(model_name, TEST_PROMPT, sample_row["review_text"]))

test_df = pl.DataFrame(test_results).with_columns(pl.lit(sample_row["label"]).alias("gold_label"))
with pl.Config(fmt_str_lengths=100):
    print(test_df)

shape: (2, 8)
┌────────────┬────────────┬──────────┬────────────┬────────────┬───────────┬───────────┬───────────┐
│ model      ┆ pred_label ┆ parse_ok ┆ raw_output ┆ elapsed_se ┆ input_tok ┆ output_to ┆ gold_labe │
│ ---        ┆ ---        ┆ ---      ┆ ---        ┆ c          ┆ ens       ┆ kens      ┆ l         │
│ str        ┆ str        ┆ bool     ┆ str        ┆ ---        ┆ ---       ┆ ---       ┆ ---       │
│            ┆            ┆          ┆            ┆ f64        ┆ i64       ┆ i64       ┆ str       │
╞════════════╪════════════╪══════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╡
│ qwen3.5:4b ┆ 배송 지연  ┆ true     ┆ {"label":  ┆ 0.47       ┆ 129       ┆ 7         ┆ 배송 지연 │
│            ┆            ┆          ┆ "배송      ┆            ┆           ┆           ┆           │
│            ┆            ┆          ┆ 지연"}     ┆            ┆           ┆           ┆           │
│ gemma4:e4b ┆ 배송 지연  ┆ true     ┆ {"label":  ┆ 0.78       ┆ 135       ┆ 10        ┆ 배송 지

# 오류 진단

- ResponseError: structured output is unavailable (status code: 501)

In [7]:
# ── 진단 설정: 출력 형식 3가지와 공통 프롬프트 ──
FORMAT_MODES = {
    "schema": OUTPUT_SCHEMA,  # JSON 스키마 강제
    "json": "json",           # JSON 형태만 강제
    "none": None,             # 강제 없음 (프롬프트 지시만)
}

# 세 방식 모두 같은 프롬프트를 써서 조건을 맞춤
DIAG_PROMPT = (
    "브라질 이커머스의 포르투갈어 불만 리뷰를 읽고, 불만의 주 원인을 "
    f"다음 라벨 중 하나로 분류하라: {', '.join(LABELS)}\n"
    '반드시 {"label": "라벨"} 형식의 JSON 한 줄로만 답하라.'
)

# ── 응답이 JSON으로 파싱되고 라벨이 7개 중 하나인지 검사 ──
def is_valid_label(raw_text):
    try:
        return json.loads(raw_text).get("label") in LABELS
    except (json.JSONDecodeError, AttributeError):
        return False

In [9]:
# ── 모델 × 형식 조합별로 호출해 결과 기록 ──
diag_records = []
for model_name, model_options in MODEL_CONFIGS.items():
    for mode_name, format_value in FORMAT_MODES.items():
        # 에러가 나도 표가 깨지지 않도록 모든 컬럼을 미리 채워둠
        record = {"model": model_name, "mode": mode_name, "status": None,
                  "valid": None, "has_thinking": None, "elapsed_sec": None, "output": None}
        try:
            start_time = time.perf_counter()
            response = ollama.chat(
                model=model_name,
                messages=[
                    {"role": "system", "content": DIAG_PROMPT},
                    {"role": "user", "content": sample_row["review_text"]},
                ],
                format=format_value,
                options={"temperature": 0},
                **model_options,
            )
            raw_text = response.message.content
            record.update(
                status="ok",
                valid=is_valid_label(raw_text),
                # 추론 텍스트가 별도 필드로 왔는지 (thinking이 실제로 꺼졌는지 확인용)
                has_thinking=bool(getattr(response.message, "thinking", None)),
                elapsed_sec=round(time.perf_counter() - start_time, 1),
                output=raw_text[:200],
            )
        except ollama.ResponseError as error:
            record.update(status=f"error {error.status_code}", output=str(error.error)[:200])
        diag_records.append(record)

# ── 결과 표 출력 (긴 문자열도 잘리지 않게) ──
diag_df = pl.DataFrame(diag_records)
with pl.Config(fmt_str_lengths=200, tbl_rows=-1):
    print(diag_df)

shape: (6, 7)
┌───────────────────┬────────┬───────────┬───────┬──────────────┬─────────────┬────────────────────┐
│ model             ┆ mode   ┆ status    ┆ valid ┆ has_thinking ┆ elapsed_sec ┆ output             │
│ ---               ┆ ---    ┆ ---       ┆ ---   ┆ ---          ┆ ---         ┆ ---                │
│ str               ┆ str    ┆ str       ┆ bool  ┆ bool         ┆ f64         ┆ str                │
╞═══════════════════╪════════╪═══════════╪═══════╪══════════════╪═════════════╪════════════════════╡
│ qwen3.5:4b        ┆ schema ┆ error 501 ┆ null  ┆ null         ┆ null        ┆ structured output  │
│                   ┆        ┆           ┆       ┆              ┆             ┆ is unavailable     │
│ qwen3.5:4b        ┆ json   ┆ error 501 ┆ null  ┆ null         ┆ null        ┆ structured output  │
│                   ┆        ┆           ┆       ┆              ┆             ┆ is unavailable     │
│ qwen3.5:4b        ┆ none   ┆ ok        ┆ true  ┆ false        ┆ 1.9        